# AIAA 3102 — Assignment 1: CIFAR-10 Image Classification

## Running environment

The submitted `.joblib` files were produced with the environment required by the
course, and this notebook is meant to be re-run in that same environment:

| Package | Version |
| --- | --- |
| Python | 3.14.6 |
| NumPy | 2.4.6 |
| SciPy | 1.18.0 |
| scikit-learn | 1.9.0 |
| joblib | 1.5.3 |

## Data paths

* `CIFAR10_Data/x_train.npy`, `CIFAR10_Data/y_train.npy` — 50,000 labelled training images
* `validation_set_to_students/images.npy`, `labels.npy` — 5,000 labelled validation images

## Final method and why it was chosen

**Features: HOG-style oriented-filter features.** Raw pixels are a weak
representation for a scikit-learn model: a pixel-space MLP tops out around 51% clean
accuracy here. The features used instead are built from a bank of steerable
derivative-of-Gaussian filters (8 orientations x 2 scales), plus two Laplacian-of-Gaussian
filters, plus per-cell colour means:

1. each image is correlated per channel with every filter kernel,
2. the absolute response is averaged over the colour channels,
3. responses are mean-pooled over a 8x8 grid of 4x4-pixel cells (HOG's cell layout),
4. the pooled vector (1,344 values) is standardised and projected onto 512 principal
   components, which whitens the features and removes the redundant filter responses.

This is the same information HOG uses, expressed with nothing but `scipy.ndimage`
correlation and NumPy pooling/slicing, so it can live inside the pickled model
(`Model_Evaluation.ipynb` only imports numpy/joblib/sklearn, so a custom transformer
class would break loading). It lifts a single model from 51% to 66% clean accuracy and
from 43% to 55% on the validation set in the same budget.

**Classifier.** An MLP (768 hidden units, Adam, `alpha=1e-4`) on the 512 PCA features.

**Ensemble.** The hidden test set mixes original and corrupted images, so a single model
trained on one data mixture has to compromise. Each submitted model is a *soft-voting
ensemble of five MLPs*, each fitted on its own mixture (the oriented-filter block is
stateless, so it is shared and computed once):

| Member | Training data | Head |
| --- | --- | --- |
| 1 | 50k clean + 30k flipped/translated | MLP 768 |
| 2 | 50k clean | MLP 768 |
| 3 | 50k clean + 30k corrupted (severity 1-3) | MLP 768 |
| 4 | 50k clean + 30k corrupted (severity 1-5) | MLP 768 |
| 5 | 50k clean + 20k flipped/translated | MLP 512-512 |

Individual members score 53-55% on the validation set; the ensemble reaches 59.1% with
seed 42. Members 3-5 also improve the heavy-corruption case, which matters because the
test corruptions may be stronger or of a different type than the validation ones.

**Augmentation.** Corruptions follow the CIFAR-10-C benchmark definitions (the 15
classical types plus the four extra CIFAR-10-C types and geometric shifts), implemented
with NumPy/SciPy only, at severities 1-3. Augmentation only affects the training data,
so it is never part of the pickled model.

**Seeds.** The pipeline, mixtures and hyper-parameters are fixed; the three submitted
models are trained from scratch with the random seeds 42, 2026 and 3102.

**Experimental workflow.** Recipes were compared on the validation set with seed 42:
raw pixels vs. oriented-filter features, cell sizes 4/8/16, 18 vs. 38 filters, with and
without concatenated raw pixels, with and without log compression, PCA dimensions
512/768, MLP widths 768/1024, and single models vs. ensembles. The configuration below
is the best of those runs; every change of the feature representation moved the result
by 4-9 points, while the remaining knobs were within noise (+-0.5), which is why the
feature choice is the core of this solution.

**Runtime.** Run All extracts the oriented-filter features for every training mixture and
trains 15 MLP pipelines; on an 18-core CPU with no GPU that takes roughly two hours
(about 40 minutes per seed). The last cell reloads every saved model from disk and
re-scores it, so the outputs already show the numbers `Model_Evaluation.ipynb` will
produce.

## 1. Environment check

In [ ]:
import os
import time

import joblib
import numpy as np
import scipy
import sklearn

print("Python  ", os.sys.version.split()[0])
print("NumPy   ", np.__version__)
print("SciPy   ", scipy.__version__)
print("sklearn ", sklearn.__version__)
print("joblib  ", joblib.__version__)

## 2. Data paths and loading

`y_train.npy` is stored as a column vector `(50000, 1)`, so it is flattened to
`(50000,)` before training.

In [ ]:
DATA_DIR = "CIFAR10_Data"
VAL_DIR = "validation_set_to_students"
MODEL_FILES = {
    42: "model_seed42.joblib",
    2026: "model_seed2026.joblib",
    3102: "model_seed3102.joblib",
}

X_train_images = np.load(os.path.join(DATA_DIR, "x_train.npy"))
y_train = np.load(os.path.join(DATA_DIR, "y_train.npy")).ravel()

X_val_images = np.load(os.path.join(VAL_DIR, "images.npy"))
y_val = np.load(os.path.join(VAL_DIR, "labels.npy"))

print("train images ", X_train_images.shape, X_train_images.dtype)
print("train labels ", y_train.shape, np.unique(y_train))
print("val   images ", X_val_images.shape, X_val_images.dtype)
print("val   labels ", y_val.shape, np.unique(y_val))

## 3. Feature extraction: oriented-filter (HOG-style) features

Everything below is built from `scipy.ndimage.convolve` and NumPy reshaping/pooling, so
the saved model contains no custom class and can be unpickled with only NumPy and
scikit-learn available.

In [ ]:
from scipy import ndimage
from sklearn.decomposition import PCA
from sklearn.ensemble import VotingClassifier
from sklearn.metrics import accuracy_score
from sklearn.neural_network import MLPClassifier
from sklearn.pipeline import FeatureUnion, Pipeline
from sklearn.preprocessing import FunctionTransformer, LabelEncoder, StandardScaler

In [ ]:
def _ft(func, **kw):
    """FunctionTransformer wrapper; the kwargs are pickled with the model."""
    return FunctionTransformer(func, kw_args=kw or None, validate=False)


def reshape(shape):
    return _ft(np.reshape, shape=shape)


def oriented_derivative_kernel(angle_deg, sigma, size=7):
    """Steerable derivative-of-Gaussian filter, shaped (1, size, size, 1)."""
    axis = np.arange(size, dtype=np.float32) - size // 2
    yy, xx = np.meshgrid(axis, axis, indexing="ij")
    gauss = np.exp(-(xx ** 2 + yy ** 2) / (2.0 * sigma ** 2))
    gx = -(xx / sigma ** 2) * gauss
    gy = -(yy / sigma ** 2) * gauss
    theta = np.deg2rad(angle_deg)
    kernel = np.cos(theta) * gx + np.sin(theta) * gy
    kernel -= kernel.mean()
    kernel /= np.abs(kernel).sum() + 1e-8
    return kernel.astype(np.float32)[None, :, :, None]


def laplacian_kernel(sigma, size=7):
    """Laplacian of Gaussian, shaped (1, size, size, 1)."""
    axis = np.arange(size, dtype=np.float32) - size // 2
    yy, xx = np.meshgrid(axis, axis, indexing="ij")
    r2 = xx ** 2 + yy ** 2
    kernel = (r2 - 2.0 * sigma ** 2) / sigma ** 4 * np.exp(-r2 / (2.0 * sigma ** 2))
    kernel -= kernel.mean()
    kernel /= np.abs(kernel).sum() + 1e-8
    return kernel.astype(np.float32)[None, :, :, None]


def filter_pool_branch(kernel, cells):
    """|filter response|, averaged over channels, pooled over a cells x cells grid."""
    size = 32 // cells
    return Pipeline([
        ("convolve", _ft(ndimage.convolve, weights=kernel)),
        ("abs", _ft(np.abs)),
        ("channel_mean", _ft(np.mean, axis=3)),
        ("reshape", reshape((-1, cells, size, cells, size))),
        ("pool", _ft(np.mean, axis=(2, 4))),
        ("flatten", reshape((-1, cells * cells))),
    ])


def colour_branch(cells):
    """Mean colour of every cell, so the model keeps the coarse colour layout."""
    size = 32 // cells
    return Pipeline([
        ("reshape", reshape((-1, cells, size, cells, size, 3))),
        ("pool", _ft(np.mean, axis=(2, 4))),
        ("flatten", reshape((-1, cells * cells * 3))),
    ])


def feature_block(cells=8, angles=8, sigmas=(1.2, 2.2), laplacians=(1.6, 3.0)):
    """Stateless raw uint8 (N, 32, 32, 3) -> pooled oriented-filter features."""
    branches = []
    for sigma in sigmas:
        for angle in np.linspace(0, 180, angles, endpoint=False):
            name = f"or{int(sigma * 10)}_{int(angle)}"
            branches.append((name, filter_pool_branch(oriented_derivative_kernel(angle, sigma), cells)))
    for sigma in laplacians:
        branches.append((f"lap{int(sigma * 10)}", filter_pool_branch(laplacian_kernel(sigma), cells)))
    branches.append(("colour", colour_branch(cells)))
    return Pipeline([
        ("as_float32", _ft(np.asarray, dtype=np.float32)),
        ("features", FeatureUnion(branches, n_jobs=1)),
    ])

## 4. Corruption and augmentation suite

CIFAR-10-C style corruptions, NumPy/SciPy only, severity 1-5. Used to build the
corrupted parts of the training set.

In [ ]:
"""CIFAR-10-C style corruption suite implemented with numpy/scipy only.

Everything here is written for the final notebook: it must run with just
Python 3.14 + numpy + scipy + scikit-learn + joblib, and no custom object
ends up inside the saved model.

Images are (N, 32, 32, 3) uint8. Each corruption takes a float batch in
[0, 1] and a severity in 1..5 (CIFAR-10-C convention) and returns float in
[0, 1].
"""

import numpy as np
from scipy import ndimage


H, W = 32, 32


def _lin(sev, lo, hi):
    """Map severity 1..5 onto the interval [lo, hi]."""
    return lo + (hi - lo) * (np.asarray(sev, dtype=np.float32) - 1.0) / 4.0


# Parameter tables follow the CIFAR-10-C benchmark (Hendrycks & Dietterich) so
# that training-time augmentations have the same severity ladder as the
# corruptions the hidden test set is built from.
GAUSS_SIGMA = [0.08, 0.12, 0.18, 0.26, 0.38]
SHOT_C = [60.0, 25.0, 12.0, 5.0, 3.0]
IMPULSE_AMOUNT = [0.03, 0.06, 0.09, 0.17, 0.27]
DEFOCUS_RADIUS = [1, 2, 3, 4, 6]
GLASS = [(0.7, 1, 1), (0.9, 2, 2), (1.1, 2, 2), (1.3, 3, 3), (1.5, 4, 4)]
MOTION = [(10, 3), (15, 5), (15, 8), (15, 12), (20, 15)]
ZOOM_MAX = [0.11, 0.16, 0.19, 0.22, 0.25]
SNOW_AMOUNT = [0.1, 0.2, 0.3, 0.4, 0.5]
FROST_SCALE = [0.5, 0.75, 1.0, 1.25, 1.5]
FOG_COVERAGE = [0.2, 0.3, 0.4, 0.5, 0.6]
BRIGHTNESS_DELTA = [0.1, 0.2, 0.3, 0.4, 0.5]
CONTRAST_FACTOR = [0.4, 0.3, 0.2, 0.1, 0.05]
ELASTIC_ALPHA = [6.0, 9.0, 12.0, 15.0, 18.0]
PIXELATE_SCALE = [0.6, 0.5, 0.4, 0.3, 0.25]
JPEG_QUALITY = [25, 18, 15, 10, 7]


def _tab(table, sev):
    return table[int(np.asarray(sev).reshape(-1)[0]) - 1]


# ---------------------------------------------------------------- noise ----
def gaussian_noise(x, sev, rng):
    sigma = _tab(GAUSS_SIGMA, sev)
    return x + rng.normal(0.0, sigma, x.shape).astype(np.float32)


def shot_noise(x, sev, rng):
    counts = _tab(SHOT_C, sev)
    return rng.poisson(np.clip(x, 0, 1) * counts) / counts


def impulse_noise(x, sev, rng):
    amount = _tab(IMPULSE_AMOUNT, sev)
    mask = rng.random(x.shape) < amount
    salt = rng.random(x.shape) < 0.5
    out = x.copy()
    out[mask] = np.where(salt[mask], 1.0, 0.0)
    return out


# ----------------------------------------------------------------- blur ----
def _disk_kernel(radius):
    r = int(radius)
    yy, xx = np.mgrid[-r : r + 1, -r : r + 1]
    k = ((yy**2 + xx**2) <= r * r).astype(np.float32)
    return k / k.sum()


def defocus_blur(x, sev, rng):
    kernel = _disk_kernel(_tab(DEFOCUS_RADIUS, sev))[None, :, :, None]
    return ndimage.convolve(x, kernel, mode="reflect")


def _line_kernel(size, angle_deg):
    k = np.zeros((size, size), np.float32)
    k[size // 2, :] = 1.0
    k = ndimage.rotate(k, angle_deg, reshape=False, order=1)
    k[k < 1e-3] = 0.0
    total = k.sum()
    return k / total if total > 0 else k


def motion_blur(x, sev, rng):
    size, angle = _tab(MOTION, sev)
    angles = np.linspace(angle, angle + 180, 6, endpoint=False)
    offsets = rng.choice(len(angles) + 1, len(x))
    out = np.empty_like(x)
    for j in range(len(angles) + 1):
        m = offsets == j
        if not m.any():
            continue
        if j == 0:
            out[m] = x[m]
        else:
            out[m] = ndimage.convolve(x[m], _line_kernel(size, angles[j - 1])[None, :, :, None], mode="reflect")
    return out


def glass_blur(x, sev, rng):
    _, delta, iters = _tab(GLASS, sev)
    d = int(delta)
    out = x
    for _ in range(int(iters)):
        off_y = rng.integers(-d, d + 1, out.shape[:-1])
        off_x = rng.integers(-d, d + 1, out.shape[:-1])
        ys = np.clip(np.arange(H)[None, :, None] + off_y, 0, H - 1)
        xs = np.clip(np.arange(W)[None, None, :] + off_x, 0, W - 1)
        rows = np.arange(len(out))[:, None, None]
        out = out[rows, ys, xs]
    return ndimage.gaussian_filter(out, sigma=(0, 0.5, 0.5, 0), mode="reflect")


def zoom_blur(x, sev, rng):
    factors = 1.0 + np.linspace(0.02, _tab(ZOOM_MAX, sev), 6)
    acc = np.zeros_like(x)
    for f in factors:
        zoomed = ndimage.zoom(x, (1, f, f, 1), order=1, mode="reflect")
        y0 = max(0, (zoomed.shape[1] - H) // 2)
        x0 = max(0, (zoomed.shape[2] - W) // 2)
        if zoomed.shape[1] >= H:
            zoomed = zoomed[:, y0 : y0 + H, x0 : x0 + W]
        else:
            pad_y = H - zoomed.shape[1]
            pad_x = W - zoomed.shape[2]
            zoomed = np.pad(
                zoomed,
                ((0, 0), (pad_y // 2, pad_y - pad_y // 2), (pad_x // 2, pad_x - pad_x // 2), (0, 0)),
                mode="edge",
            )
        acc += zoomed
    return acc / len(factors)


# -------------------------------------------------------------- weather ----
def fog(x, sev, rng):
    strength = _tab(FOG_COVERAGE, sev) / 1.4
    small = rng.normal(0.6, 0.25, (len(x), 4, 4, 3)).astype(np.float32)
    mask = ndimage.zoom(small, (1, H / 4, W / 4, 1), order=1, mode="reflect")
    mask = np.clip(mask, 0, 1)
    fog_color = np.array([0.75, 0.78, 0.80], dtype=np.float32)
    alpha = (mask * strength).astype(np.float32)
    return x * (1 - alpha) + fog_color * alpha


def snow(x, sev, rng):
    dep = _tab(SNOW_AMOUNT, sev) / 1.6
    layer = np.zeros_like(x)
    for _ in range(3):
        flakes = rng.normal(0, 0.06 + 0.12 * dep, x.shape).astype(np.float32)
        flakes[flakes < 0.25] = 0
        layer += ndimage.gaussian_filter(flakes, sigma=(0, 0.6, 0.6, 0), mode="reflect")
    return x * (1 - dep) + (x + layer) * dep


def frost(x, sev, rng):
    pattern = rng.normal(0, 1, (len(x), 4, 4, 3)).astype(np.float32)
    pattern = ndimage.zoom(pattern, (1, H / 4, W / 4, 1), order=1, mode="reflect")
    pattern = ndimage.gaussian_filter(np.abs(pattern), sigma=(0, 1.2, 1.2, 0), mode="reflect")
    pattern = np.clip(pattern * _tab(FROST_SCALE, sev), 0, 1)
    return x * (1 - pattern * 0.6) + pattern * 0.6


# ------------------------------------------------------------- digital ----
def brightness(x, sev, rng):
    delta = _tab(BRIGHTNESS_DELTA, sev)
    sign = rng.choice([-1.0, 1.0])
    return x + sign * delta


def contrast(x, sev, rng):
    factor = _tab(CONTRAST_FACTOR, sev)
    mean = x.mean(axis=(1, 2, 3), keepdims=True)
    return (x - mean) * factor + mean


def pixelate(x, sev, rng):
    scale = _tab(PIXELATE_SCALE, sev)
    n = max(2, int(round(H * scale)))
    small = ndimage.zoom(x, (1, n / H, n / W, 1), order=1, mode="reflect")
    return ndimage.zoom(small, (1, H / n, W / n, 1), order=0, mode="reflect")


_JPEG_LUMA = np.array(
    [
        [16, 11, 10, 16, 24, 40, 51, 61],
        [12, 12, 14, 19, 26, 58, 60, 55],
        [14, 13, 16, 24, 40, 57, 69, 56],
        [14, 17, 22, 29, 51, 87, 80, 62],
        [18, 22, 37, 56, 68, 109, 103, 77],
        [24, 35, 55, 64, 81, 104, 113, 92],
        [49, 64, 78, 87, 103, 121, 120, 101],
        [72, 92, 95, 98, 112, 100, 103, 99],
    ],
    dtype=np.float32,
)


def jpeg_at_quality(x, quality):
    """JPEG round-trip at an explicit quality factor (no severity ladder)."""
    from scipy.fft import dctn, idctn

    quality = float(quality)
    scale = 5000.0 / quality if quality < 50 else 200.0 - 2.0 * quality
    table = _JPEG_LUMA * scale / 100.0
    table_chroma = table * 1.4

    y = x * 255.0 - 128.0
    r, g, b = y[..., 0], y[..., 1], y[..., 2]
    yc = 0.299 * r + 0.587 * g + 0.114 * b
    cb = -0.168736 * r - 0.331264 * g + 0.5 * b + 128.0
    cr = 0.5 * r - 0.418688 * g - 0.081312 * b + 128.0
    stacked = np.stack([yc, cb - 128.0, cr - 128.0], -1)

    blocks = stacked.reshape(len(x), H // 8, 8, W // 8, 8, 3).transpose(0, 1, 3, 5, 2, 4)
    coef = dctn(blocks, axes=(-2, -1), norm="ortho")
    q = np.stack([table, table_chroma, table_chroma])  # (3, 8, 8)
    q = np.repeat(q[None], len(x) * (H // 8) * (W // 8), axis=0).reshape(coef.shape)
    coef = np.round(coef / q) * q
    blocks = idctn(coef, axes=(-2, -1), norm="ortho")
    stacked = blocks.transpose(0, 1, 4, 2, 5, 3).reshape(len(x), H, W, 3)

    yc, cb, cr = stacked[..., 0], stacked[..., 1] + 128.0, stacked[..., 2] + 128.0
    r = yc + 1.402 * (cr - 128.0)
    g = yc - 0.344136 * (cb - 128.0) - 0.714136 * (cr - 128.0)
    b = yc + 1.772 * (cb - 128.0)
    return (np.stack([r, g, b], -1) + 128.0) / 255.0


def jpeg_compression(x, sev, rng):
    return jpeg_at_quality(x, _tab(JPEG_QUALITY, sev))


def elastic(x, sev, rng):
    alpha = _tab(ELASTIC_ALPHA, sev)
    disp = rng.normal(0, 1, (len(x), 4, 4, 2)).astype(np.float32)
    disp = ndimage.zoom(disp, (1, H / 4, W / 4, 1), order=1, mode="reflect")
    disp = ndimage.gaussian_filter(disp, sigma=(0, 2.0, 2.0, 0), mode="reflect") * alpha
    yy, xx = np.meshgrid(np.arange(H), np.arange(W), indexing="ij")
    coords = np.stack(
        [
            np.clip(yy[None] + disp[..., 0], 0, H - 1.001),
            np.clip(xx[None] + disp[..., 1], 0, W - 1.001),
        ],
        axis=-1,
    )
    return _bilinear(x, coords)


def _bilinear(images, coords):
    y, x = coords[..., 0], coords[..., 1]
    y0, x0 = np.floor(y).astype(np.int32), np.floor(x).astype(np.int32)
    y1, x1 = np.minimum(y0 + 1, H - 1), np.minimum(x0 + 1, W - 1)
    wy, wx = (y - y0)[..., None], (x - x0)[..., None]
    rows = np.arange(len(images))[:, None, None]
    tl = images[rows, y0, x0]
    tr = images[rows, y0, x1]
    bl = images[rows, y1, x0]
    br = images[rows, y1, x1]
    top = tl * (1 - wx) + tr * wx
    bottom = bl * (1 - wx) + br * wx
    return top * (1 - wy) + bottom * wy


SPECKLE_C = [0.06, 0.1, 0.15, 0.2, 0.25]
GAUSSIAN_BLUR_SIGMA = [1.0, 1.4, 1.9, 2.6, 3.6]
SPATTER_AMOUNT = [0.02, 0.04, 0.07, 0.11, 0.16]
SATURATE_FACTOR = [1.15, 1.5, 2.0, 2.6, 3.2]
TRANSLATE_FRAC = [0.06, 0.1, 0.15, 0.2, 0.27]
ROTATE_DEG = [10.0, 15.0, 20.0, 25.0, 30.0]


def speckle_noise(x, sev, rng):
    c = _tab(SPECKLE_C, sev)
    return x + x * rng.normal(0, c, x.shape).astype(np.float32)


def gaussian_blur(x, sev, rng):
    sigma = _tab(GAUSSIAN_BLUR_SIGMA, sev)
    return ndimage.gaussian_filter(x, sigma=(0, sigma, sigma, 0), mode="reflect")


def spatter(x, sev, rng):
    """Dark mud-like blobs, the classic CIFAR-10-C spatter look."""
    amount = _tab(SPATTER_AMOUNT, sev)
    out = x.copy()
    for i in range(len(x)):
        n = rng.integers(3, 7)
        for _ in range(n):
            cy, cx = rng.integers(0, 32, 2)
            r = rng.integers(2, 6)
            yy, xx = np.mgrid[0:32, 0:32]
            m = ((yy - cy) ** 2 + (xx - cx) ** 2) <= r * r
            out[i][m] = out[i][m] * (1 - amount * 4) + np.array([0.25, 0.2, 0.15]) * amount * 4
    return out


def saturate(x, sev, rng):
    factor = _tab(SATURATE_FACTOR, sev)
    gray = (x * np.array([0.299, 0.587, 0.114], np.float32)).sum(-1, keepdims=True)
    return gray + (x - gray) * factor


def translate(x, sev, rng):
    frac = _tab(TRANSLATE_FRAC, sev)
    out = np.zeros_like(x)
    for i in range(len(x)):
        dy = int(rng.integers(-32, 33) * frac)
        dx = int(rng.integers(-32, 33) * frac)
        y0, y1 = max(0, dy), min(H, H + dy)
        x0, x1 = max(0, dx), min(W, W + dx)
        out[i, y0:y1, x0:x1] = x[i, y0 - dy : y1 - dy, x0 - dx : x1 - dx]
    return out


def rotate(x, sev, rng):
    deg = _tab(ROTATE_DEG, sev)
    out = np.empty_like(x)
    for i in range(len(x)):
        angle = rng.uniform(-deg, deg)
        out[i] = ndimage.rotate(x[i], angle, reshape=False, order=1, mode="constant", cval=0.0)
    return out


CORRUPTIONS = {
    "gaussian_noise": gaussian_noise,
    "shot_noise": shot_noise,
    "impulse_noise": impulse_noise,
    "defocus_blur": defocus_blur,
    "glass_blur": glass_blur,
    "motion_blur": motion_blur,
    "zoom_blur": zoom_blur,
    "snow": snow,
    "frost": frost,
    "fog": fog,
    "brightness": brightness,
    "contrast": contrast,
    "elastic": elastic,
    "pixelate": pixelate,
    "jpeg": jpeg_compression,
    "speckle_noise": speckle_noise,
    "gaussian_blur": gaussian_blur,
    "spatter": spatter,
    "saturate": saturate,
    "translate": translate,
    "rotate": rotate,
}

CORRUPTION_NAMES = list(CORRUPTIONS)


def corrupt_batch(images_uint8, name, sev, rng):
    """Apply one corruption to a uint8 batch; returns uint8."""
    x = images_uint8.astype(np.float32) / 255.0
    out = CORRUPTIONS[name](x, sev, rng)
    return np.clip(out * 255.0, 0, 255).astype(np.uint8)


def make_all(images_uint8, sev, rng):
    """Pick a random corruption type per image at a fixed severity."""
    out = np.empty_like(images_uint8)
    choice = rng.integers(0, len(CORRUPTION_NAMES), len(images_uint8))
    for k in np.unique(choice):
        m = choice == k
        out[m] = corrupt_batch(images_uint8[m], CORRUPTION_NAMES[k], sev, rng)
    return out

## 5. Building the training mixtures

In [ ]:
RECIPE = {
    "n_clean": 50000,          # all original training images
    "n_shift": 30000,          # flipped / translated copies
    "n_shift2": 20000,         # second, smaller translated copy set
    "n_corr": 30000,           # corrupted copies (used twice, at severities 1-3 and 1-5)
    "pca": 512,
    "epochs": 55,
    "alpha": 1e-4,
    "seeds": [42, 2026, 3102],
    # (name, mixture, hidden layers)
    "members": [
        ("clean+shift", ("clean", "shift"), (768,)),
        ("clean", ("clean",), (768,)),
        ("clean+corrupted13", ("clean", "corr13"), (768,)),
        ("clean+corrupted15", ("clean", "corr15"), (768,)),
        ("clean+shift-2layer", ("clean", "shift2"), (512, 512)),
    ],
}


def random_flip_shift(images, rng, max_shift=3):
    """Horizontal flip plus a small translation (edge-clamped)."""
    x = images.copy()
    flip = rng.random(len(x)) < 0.5
    if flip.any():
        x[flip] = x[flip][:, :, ::-1]
    dy = rng.integers(-max_shift, max_shift + 1, len(x))
    dx = rng.integers(-max_shift, max_shift + 1, len(x))
    rows = np.arange(len(x))[:, None, None]
    ys = np.clip(np.arange(32)[None, :, None] + dy[:, None, None], 0, 31)
    xs = np.clip(np.arange(32)[None, None, :] + dx[:, None, None], 0, 31)
    return x[rows, ys, xs]


def make_corrupted(images, rng, sev_lo, sev_hi):
    """Give every image a random corruption type and a random severity."""
    out = np.empty_like(images)
    choice = rng.integers(0, len(CORRUPTION_NAMES), len(images))
    sev = rng.integers(sev_lo, sev_hi + 1, len(images))
    for k in np.unique(choice):
        for s in np.unique(sev[choice == k]):
            m = (choice == k) & (sev == s)
            out[m] = corrupt_batch(images[m], CORRUPTION_NAMES[k], int(s), rng)
    return out


def build_model(seed, X_pool, y_pool, recipe):
    """Train one submitted model: soft-voting ensemble of five MLP pipelines."""
    t0 = time.time()
    rng = np.random.default_rng(seed)
    feats = feature_block()

    idx_shift = rng.choice(len(X_pool), recipe["n_shift"], replace=False)
    idx_shift2 = rng.choice(len(X_pool), recipe["n_shift2"], replace=False)
    idx_corr13 = rng.choice(len(X_pool), recipe["n_corr"], replace=False)
    idx_corr15 = rng.choice(len(X_pool), recipe["n_corr"], replace=False)

    images = {
        "clean": X_pool,
        "shift": random_flip_shift(X_pool[idx_shift], rng),
        "shift2": random_flip_shift(X_pool[idx_shift2], rng),
        "corr13": make_corrupted(X_pool[idx_corr13], rng, 1, 3),
        "corr15": make_corrupted(X_pool[idx_corr15], rng, 1, 5),
    }
    labels = {
        "clean": y_pool,
        "shift": y_pool[idx_shift],
        "shift2": y_pool[idx_shift2],
        "corr13": y_pool[idx_corr13],
        "corr15": y_pool[idx_corr15],
    }

    # the feature block is stateless: extract once, reuse for every member
    features = {}
    for name, images_part in images.items():
        features[name] = feats.transform(images_part).astype(np.float32)
        print(f"    features {name:7s} {features[name].shape}  {time.time() - t0:.0f}s", flush=True)
    del images

    members = []
    for name, keys, hidden in recipe["members"]:
        tm = time.time()
        X_member = np.concatenate([features[k] for k in keys])
        y_member = np.concatenate([labels[k] for k in keys])
        inner = Pipeline([
            ("standardize", StandardScaler()),
            ("pca", PCA(n_components=recipe["pca"], random_state=seed)),
            (
                "mlp",
                MLPClassifier(
                    hidden_layer_sizes=hidden,
                    alpha=recipe["alpha"],
                    batch_size=256,
                    learning_rate_init=1e-3,
                    max_iter=recipe["epochs"],
                    random_state=seed,
                ),
            ),
        ]).fit(X_member, y_member)
        members.append(Pipeline([("features", feats), ("inner", inner)]))
        print(
            f"    member {name:20s} {X_member.shape[0]:6d} samples, "
            f"{inner.named_steps['mlp'].n_iter_} epochs, {time.time() - tm:.0f}s",
            flush=True,
        )
        del X_member, y_member

    ensemble = VotingClassifier([(f"member{i}", m) for i, m in enumerate(members)], voting="soft")
    # the members are already fitted on their own mixtures
    ensemble.estimators_ = members
    ensemble.le_ = LabelEncoder().fit(y_pool)
    ensemble.classes_ = ensemble.le_.classes_
    return ensemble

## 6. Train and save the three models

Each seed rebuilds every training mixture, re-extracts the features and retrains all five
members from scratch.

In [ ]:
summary = []
for seed in RECIPE["seeds"]:
    t0 = time.time()
    print(f"seed {seed}: building training data")
    X_pool = X_train_images[: RECIPE["n_clean"]]
    y_pool = y_train[: RECIPE["n_clean"]]
    model = build_model(seed, X_pool, y_pool, RECIPE)
    val_acc = accuracy_score(y_val, model.predict(X_val_images))
    path = MODEL_FILES[seed]
    joblib.dump(model, path)
    size_mb = os.path.getsize(path) / 1e6
    summary.append((seed, val_acc, size_mb, time.time() - t0))
    print(
        f"seed {seed}: validation accuracy {val_acc:.4f}  |  "
        f"{size_mb:.1f} MB  |  {time.time() - t0:.0f}s",
        flush=True,
    )

## 7. Verification: reload the saved models

In [ ]:
accuracies = []
for seed, path in MODEL_FILES.items():
    model = joblib.load(path)
    acc = accuracy_score(y_val, model.predict(X_val_images))
    accuracies.append(acc)
    print(f"{path}: validation accuracy {acc:.4f}  ({os.path.getsize(path) / 1e6:.1f} MB)")

print()
print("Individual accuracies:")
for (seed, _, size_mb, secs), acc in zip(summary, accuracies):
    print(f"  model_seed{seed}.joblib: {acc:.4f}  ({size_mb:.1f} MB, {secs:.0f}s)")
print(f"\nAverage validation accuracy: {np.mean(accuracies):.4f}")
print(f"All files below 100 MB: {all(os.path.getsize(p) / 1e6 < 100 for p in MODEL_FILES.values())}")